# Step 4 · Planarian context and human baseline comparison

Each step presents the planarian analysis first, followed by the separate human analysis.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT=next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'dataset').is_dir() and (p/'results').is_dir()), None)
assert ROOT is not None, 'Start Jupyter within the repository'
assert (ROOT/'dataset').is_dir() and (ROOT/'results').is_dir(), 'Start Jupyter from the repository root or notebooks/'
sys.path.insert(0,str(ROOT))
PROJECT=ROOT/'dataset'
PANDO=ROOT/'results/pando'
MODEL=ROOT/'results/model'
SCENIC=ROOT/'results/scenic'
SCENIC_DATA=ROOT/'dataset/scenic'
pd.set_option('display.max_columns',30)

print('Package:',ROOT)
print('Python:',sys.version.split()[0])
print('Directories available:', all(p.exists() for p in [PROJECT,PANDO,MODEL,SCENIC]))

## Planarian · comparison context

The package contains the planarian model outputs above, but no comparable planarian Pando/SCENIC benchmark for this analysis. A planarian baseline comparison is therefore not reported. The following comparison uses the separately analyzed human brain organoid data.


In [ ]:
planarian_top=pd.read_csv(ROOT/'results/planarian/phase3_output/Hes1_top10.tsv',sep='\t')
print('Planarian example regulator:',planarian_top.source.iloc[0],'ranked candidates:',len(planarian_top))
print('Comparable planarian Pando/SCENIC baseline: not available')


## Human · Pando and SCENIC comparison

Only after preparing, training and presenting pando+GRN do we inspect the Pando baseline and the SCENIC workflow. The input features of pando+GRN include GRNBoost2 importance; consequently the compared methods do not use independent inputs. This comparison is descriptive and targeted, not a blinded test of superiority.

## Complete SCENIC baseline

SCENIC includes GRNBoost2 network inference, motif-based cisTarget pruning, and AUCell. The default workflow finished for 9,327 sampled cells and produced 377 regulons; no GLI3 regulon passed NES ≥ 3.0. The subsequent Top-10 table therefore reports the **GRNBoost2 stage before motif filtering**, clearly distinguished from complete SCENIC.

In [ ]:
full=SCENIC/'full'
pruning=json.loads((full/'pruning_summary.json').read_text()); auinfo=json.loads((full/'aucell_summary.json').read_text())
print('Pruning:',pruning); print('AUCell:',auinfo)
auc=pd.read_csv(full/'aucell_matrix.csv.gz',index_col=0)
cells=pd.read_csv(SCENIC_DATA/'cells.csv')
assert auc.shape==(9327,377) and list(auc.index)==cells.cell.tolist() and not auc.isna().any().any()
assert pruning['GLI3_regulons']==[]
print('AUCell shape:',auc.shape,'GLI3 columns:',[x for x in auc if x.startswith('GLI3(')])

In [ ]:
reg=pd.read_csv(full/'regulon_targets.tsv.gz',sep='\t')
for tf in ['GLI3','HES5','PAX6']:
 rows=reg[reg.TF.eq(tf)]
 print(tf,':',rows.regulon.nunique(),'regulons;',rows.target.nunique(),'distinct targets')

`HES5(+)` and `PAX6(+)` represent regulons where those genes act as regulators; they do **not** show that GLI3 regulates HES5 or PAX6. The printed SCENIC comparator in the later Top-10 table is the *pre-motif GRNBoost2 ranking*; complete SCENIC has no GLI3 Top-10 result. Lower-threshold post-hoc exploratory tables are in `results/scenic/motif_threshold_analysis/` and must not be treated as default SCENIC findings.

## Common-universe GLI3 comparison

We use the updated candidate table and 1,794-gene universe; 48 candidate genes fall within this universe. HES4 was added post hoc. Ranks follow each method's own saved scoring and filtering rules.

In [ ]:
audit=pd.read_csv(PROJECT/'inputs/target_scores.tsv',sep='\t')
genes=set((SCENIC_DATA/'genes.txt').read_text().splitlines())
universe=(set(audit.target)|{'HES4'}) & genes
candidates=pd.read_csv(PROJECT/'inputs/PRIORITY_with_strong_local_DA_peak.tsv',sep='\t')
positive=set(candidates.gene_symbol.dropna()) & universe
print('Common genes:',len(universe),'candidates in table:',len(candidates),'candidates in common:',len(positive))
assert len(universe)==1794 and len(positive)==48

In [ ]:
raw=pd.read_csv(PROJECT/'inputs/TF_gene_network_us.csv')
external=pd.read_csv(PROJECT/'inputs/metacells.csv')
boost=pd.read_csv(SCENIC/'adjacencies.tsv',sep='\t')
def pando_rank(df,p_col):
 x=df[df.tf.eq('GLI3') & df.target.isin(universe) & df[p_col].between(0,.05)].copy()
 x['magnitude']=x.estimate.abs()
 return x.sort_values([p_col,'magnitude','target'],ascending=[True,False,True]).drop_duplicates('target').rename(columns={p_col:'score'})
methods={
 'Our Pando':pando_rank(raw,'padj_min'),
 'Pando baseline':pando_rank(external,'padj'),
 'SCENIC (before motif filtering)':boost[boost.TF.eq('GLI3') & boost.target.isin(universe)].rename(columns={'importance':'score'}).sort_values(['score','target'],ascending=[False,True]).drop_duplicates('target'),
 'pando+GRN':pd.read_csv(MODEL/'GLI3_all_scores.tsv',sep='\t').sort_values(['score','target'],ascending=[False,True]),
}
rows=[];tops=[]
for label,frame in methods.items():
 x=frame.reset_index(drop=True).copy();x['rank']=np.arange(1,len(x)+1);x['hit']=x.target.isin(positive);t=x.head(10).copy();t.insert(0,'method',label);tops.append(t[['method','rank','target','score','hit']]); ranks=dict(zip(x.target,x['rank']))
 rows.append({'method':label,'ranked_targets':len(x),'hits_top10':int(t.hit.sum()),'hit_genes':', '.join(t.loc[t.hit,'target']),'HES4_rank':ranks.get('HES4'),'HES5_rank':ranks.get('HES5')})
comparison=pd.DataFrame(rows); top=pd.concat(tops,ignore_index=True)
print(comparison[comparison.method.ne('Our Pando')].to_string(index=False))


In [ ]:
output=ROOT/'results/generated';output.mkdir(exist_ok=True)
comparison[comparison.method.ne('Our Pando')].to_csv(output/'four_step_comparison.tsv',sep='\t',index=False)
top[top.method.ne('Our Pando')].to_csv(output/'four_step_top10.tsv',sep='\t',index=False)
print('Saved:',output/'four_step_comparison.tsv')

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(15,5)); order=['pando+GRN','Pando baseline','SCENIC (before motif filtering)']
for row,method in enumerate(order):
 x=top[top.method.eq(method)]; ax.scatter(x['rank'],np.full(len(x),row),c=np.where(x.hit,'#087e76','white'),edgecolors=np.where(x.hit,'#087e76','#a8b7bf'),s=65)
 for item in x.itertuples(): ax.annotate(item.target,(item.rank,row),xytext=(0,8),textcoords='offset points',rotation=35,ha='left',va='bottom',fontsize=8,color='#087e76' if item.hit else '#182b38')
 ax.text(10.7,row,f'{int(x.hit.sum())}/10',va='center',weight='bold')
ax.set(yticks=range(3),yticklabels=order,xticks=range(1,11),xlabel='Rank',xlim=(.5,11.4),ylim=(2.4,-.7),title='GLI3 candidate hits (exploratory)')
for sp in ax.spines.values():sp.set_visible(False)
fig.tight_layout();fig.savefig(output/'four_step_top10.png',dpi=180,bbox_inches='tight');plt.show()

The pando+GRN row ranks all candidate targets, whereas Pando baselines rank filtered edges; score values are not directly comparable. The existing publication-style plot is `results/figures/gli3_reviewer_dot_matrix.pdf`.

## Optional SCENIC re-execution

The full stage scripts and checkpoints are included. In a disposable copy with a suitable SCENIC environment, set `RUN_SCENIC_STAGE` to `grn`, `modules`, `prune`, or `aucell`; the corresponding stage will run and may overwrite its checkpoint/output directory. See `docs/reproducibility.md` before using this option.

In [ ]:
import os, subprocess
stages={name:ROOT/'src/scenic'/script for name,script in {'grn':'grn.py','modules':'prepare_modules.py','prune':'prune_batches.py','aucell':'run_aucell.py'}.items()}
stage=os.environ.get('RUN_SCENIC_STAGE','none')
if stage=='none':
 print('SCENIC recomputation skipped; set RUN_SCENIC_STAGE to one of:',', '.join(stages))
else:
 assert stage in stages
 subprocess.run([sys.executable,str(stages[stage])],check=True,cwd=ROOT)